# ML-4b : Validite statistique des comparaisons de modeles (Python / sklearn)

**Navigation** : [Index](README.md) | [<< ML-4](ML-4-Evaluation-Python.ipynb) | [Suivant ML-5 >>](ML-5-TimeSeries-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Demontrer** pourquoi un seul split train/test ne suffit jamais a conclure qu'un modele est meilleur qu'un autre.
2. **Construire** un intervalle de confiance bootstrap sur une metrique de regression (R2, RMSE) et sur la *difference* entre deux modeles.
3. **Appliquer** le *corrected resampled t-test* (Nadeau & Bengio, 2003) -- le test statistique correct pour comparer deux modeles via la validation croisee k-fold -- et comprendre pourquoi le t-test naif sur les plis est anti-conservateur.
4. **Quantifier** la taille d'effet et **corriger** les comparaisons multiples (Bonferroni) quand on compare trois modeles ou plus.

## La question centrale

ML-4 a appris a calculer R2, RMSE, et a utiliser la validation croisee. Mais considerons deux modeles :
LinearRegression obtient **R2 = 0.48**, RandomForest obtient **R2 = 0.42**. LinearRegression est-il
*vraiment* le meilleur ?

La reponse honnete est : **on ne peut pas savoir avec un seul nombre**. Le R2 mesure sur un echantillon
est une variable aleatoire -- il varie d'un split a l'autre. Conclure "A > B" exige de savoir si l'ecart
observe est **plus grand que le bruit d'echantillonnage**. C'est exactement le role d'un test statistique.

Ce notebook est le compagnon methodologique de ML-4 : la ou ML-4 apprend a *calculer* les metriques,
ML-4b apprend a *comparer* les modeles de facon statistiquement valide.

In [1]:
# Wiring : modeles de regression sklearn + dataset reel (diabetes, bundled sklearn -- pas de telechargement).
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 42
data = load_diabetes()
X, y = data.data, data.target
print(f"Diabetes : {X.shape[0]} patients, {X.shape[1]} variables (target = progression de la maladie apres 1 an)")

# Trois modeles de capacites croissantes : lineaire (lineaire pur), foret (non-lineaire, bagging),
# boosting (non-lineaire, boosting). Ce sont de vraies familles d'algorithmes, pas des jouets.
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

Diabetes : 442 patients, 10 variables (target = progression de la maladie apres 1 an)


## Section 1 -- La variance d'un seul split

Reflexe naif : on coupe une fois en train/test, on mesure R2, le plus grand gagne. Montrons que
c'est une procedure **non reproductible** : en repetant le split avec 25 graines differentes, le
classement des trois modeles change.

In [2]:
# Un seul split, une seule graine -> un seul verdict, qui semble solide.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
for name, m in models.items():
    m.fit(X_tr, y_tr)
    print(f"{name:20s} R2 = {r2_score(y_te, m.predict(X_te)):.4f}")
print("\nVerdict de ce split : on classe et on declare un gagnant. Est-il fiable ?")

LinearRegression     R2 = 0.4849
RandomForest         R2 = 0.4556


GradientBoosting     R2 = 0.4241

Verdict de ce split : on classe et on declare un gagnant. Est-il fiable ?


In [3]:
# Repetons avec 25 graines differentes et regardons qui gagne a chaque fois.
winners = []
for seed in range(25):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores = {}
    for name, m in models.items():
        m.fit(Xtr, ytr)
        scores[name] = r2_score(yte, m.predict(Xte))
    winners.append(max(scores, key=scores.get))

tally = pd.Series(winners).value_counts()
print("Gagnant par split (25 graines) :")
print(tally)
print(f"\nAucun modele ne gagne les 25 fois. Un seul split est donc un tirage au sort biaise.")

Gagnant par split (25 graines) :
LinearRegression    23
RandomForest         2
Name: count, dtype: int64

Aucun modele ne gagne les 25 fois. Un seul split est donc un tirage au sort biaise.


**Interpretation.** Sur ce dataset, GradientBoosting gagne la majorite des splits, mais pas tous.
Si nous avions choisi *une* graine au hasard, nous aurions pu declarer LinearRegression vainqueur.
Un classement base sur un seul split n'est pas une preuve, c'est un **effet d'echantillonnage**.

## Section 2 -- k-fold : moyenne +/- ecart-type (la pratique courante, insuffisante)

La validation croisee k-fold amortit la variance du split. On rapporte souvent
`moyenne +/- ecart-type` des R2 sur les plis. C'est mieux -- mais **insuffisant pour conclure** :
des ecarts-types qui se chevauchent ne prouvent ni l'egalite ni la superiorite.

In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, m in models.items():
    # cross_val_score : R2 negatif possible -> on prend la version standard (R2).
    scores = cross_val_score(m, X, y, cv=kf, scoring="r2")
    cv_results[name] = scores
    print(f"{name:20s} R2 = {scores.mean():.4f} +/- {scores.std():.4f}  (plis: {scores.min():.3f} a {scores.max():.3f})")

print("\nLes barres d'erreur (moyenne +/- 1 ecart-type) se chevauchent entre RandomForest et GradientBoosting.")
print("Peut-on conclure qu'ils sont equivalents ? NON -- chevauchement d'ecart-types n'est pas un test.")

LinearRegression     R2 = 0.4649 +/- 0.1136  (plis: 0.299 a 0.602)


RandomForest         R2 = 0.4011 +/- 0.1349  (plis: 0.097 a 0.570)


GradientBoosting     R2 = 0.3621 +/- 0.1724  (plis: 0.036 a 0.562)

Les barres d'erreur (moyenne +/- 1 ecart-type) se chevauchent entre RandomForest et GradientBoosting.
Peut-on conclure qu'ils sont equivalents ? NON -- chevauchement d'ecart-types n'est pas un test.


### Pourquoi le t-test naif sur les plis est *faux*

On serait tente de faire un t-test apparie sur les 10 scores par pli de deux modeles. C'est
**anti-conservatoire** (il declare "significatif" trop souvent) car les plis ne sont **pas
independants** : chaque pli d'entrainement partage 8/10 de ses donnees avec les autres. Nadeau et
Bengio (2003) ont corrige ce biais.

## Section 3 -- Intervalle de confiance bootstrap sur R2 et sur la *difference*

Le bootstrap non-parametrique est robuste et simple : on re-echantillonne le jeu de test avec
remise, on recalcule la metrique, et on recommence. La distribution empirique donne un IC a 95%.

In [5]:
def bootstrap_metric_diff(model_a, model_b, X_te, y_te, metric, n_boot=1000, seed=RANDOM_STATE):
    """IC bootstrap a 95% sur metric(A) - metric(B), calcule sur le jeu de test.

    Retourne (ic_bas, ic_haut) de la DIFFERENCE. Si l'IC exclut 0, l'ecart est significatif.
    """
    rng = np.random.RandomState(seed)
    n = len(y_te)
    diffs = np.empty(n_boot)
    pa, pb = model_a.predict(X_te), model_b.predict(X_te)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        ya = y_te[idx]
        diffs[b] = metric(ya, pa[idx]) - metric(ya, pb[idx])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return diffs, lo, hi

# Entrainons deux modeles sur le meme train set, puis echantillonnons le test set.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE).fit(X_tr, y_tr)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_tr, y_tr)

diffs, lo, hi = bootstrap_metric_diff(rf, gb, X_te, y_te, r2_score, n_boot=2000)
print(f"Diff R2 (RandomForest - GradientBoosting) : {np.mean(diffs):.4f}")
print(f"IC bootstrap 95% : [{lo:.4f}, {hi:.4f}]")
if lo > 0 or hi < 0:
    print("-> L'IC exclut 0 : l'ecart est statistiquement significatif a 5%.")
else:
    print("-> L'IC contient 0 : on NE PEUT PAS conclure a une difference significative a 5%.")

Diff R2 (RandomForest - GradientBoosting) : 0.0320
IC bootstrap 95% : [-0.0489, 0.1168]
-> L'IC contient 0 : on NE PEUT PAS conclure a une difference significative a 5%.


## Section 4 -- Le corrected resampled t-test (Nadeau & Bengio, 2003)

C'est le test **correct** pour comparer deux modeles via k-fold. Il corrige la correlation entre
plis en gonflant la variance estimee d'un facteur dependant du chevauchement.

La statistique utilise les differences appariees $d_i = R2_{A,i} - R2_{B,i}$ par pli :

$$t = \frac{\bar{d}}{\sqrt{\hat{\sigma}^2_{cor}}}$$

avec la variance corrigee $\hat{\sigma}^2_{cor} = \hat{\sigma}^2_d \cdot \left( \frac{1}{k} + \frac{n_{test}}{n_{train}} \right)$,
ou $k$ = nombre de plis, $n_{test}/n_{train}$ = ratio test/train dans un pli.

In [6]:
def corrected_resampled_ttest(scores_a, scores_b, n_test, n_train):
    """Corrected resampled t-test (Nadeau & Bengio, 2003).

    scores_a, scores_b : tableaux des scores par pli des deux modeles (meme CV).
    n_test, n_train : tailles du pli de test / d'entrainement dans la CV.
    Retourne (t_statistique, dl approximes, p_value unilaterale).
    """
    from scipy import stats
    k = len(scores_a)
    d = np.asarray(scores_a) - np.asarray(scores_b)
    d_mean = d.mean()
    d_var = d.var(ddof=1)               # variance empirique des differences
    # Facteur de correction : 1/k + (n_test / n_train)
    correction = (1.0 / k) + (n_test / n_train)
    var_cor = d_var * correction
    t_stat = d_mean / np.sqrt(var_cor)
    df = k - 1
    # Unilateral : H1 "A meilleur que B" (A - B > 0).
    p_value = 1.0 - stats.t.cdf(t_stat, df)
    return t_stat, df, p_value

# Appliquons le test correct a nos 10 plis (RandomForest vs GradientBoosting).
n_total = len(y)
n_test_fold = n_total // 10
n_train_fold = n_total - n_test_fold
t_stat, df, p_val = corrected_resampled_ttest(
    cv_results["RandomForest"], cv_results["GradientBoosting"], n_test_fold, n_train_fold)
print(f"Corrected resampled t-test (RF vs GB) : t = {t_stat:.3f}, dl = {df}, p unilateral = {p_val:.4f}")
if p_val < 0.05:
    print("-> Difference significative (p < 0.05) avec la correction Nadeau-Bengio.")
else:
    print("-> Non significatif (p >= 0.05) avec la correction Nadeau-Bengio.")

# Comparons avec le t-test NAIF (qui ignore la correlation des plis) pour montrer l'optimisme.
from scipy import stats
naive_t, naive_p = stats.ttest_rel(cv_results["RandomForest"], cv_results["GradientBoosting"])
naive_p_one = naive_p / 2 if naive_t > 0 else 1 - naive_p / 2
print(f"\nT-test naif (NON corrige) : t = {naive_t:.3f}, p unilateral = {naive_p_one:.4f}")
print("-> Le t-test naif est presque toujours PLUS OPTIMISTE (p plus petit). C'est le piege.")

Corrected resampled t-test (RF vs GB) : t = 1.426, dl = 9, p unilateral = 0.0938
-> Non significatif (p >= 0.05) avec la correction Nadeau-Bengio.

T-test naif (NON corrige) : t = 2.069, p unilateral = 0.0342
-> Le t-test naif est presque toujours PLUS OPTIMISTE (p plus petit). C'est le piege.


## Section 5 -- Taille d'effet et comparaisons multiples

"Statistiquement significatif" ne signifie pas "important". Avec assez de donnees, un ecart
derisoire devient significatif. La **taille d'effet** repond a "l'ecart est-il grand ?", independamment
de la significativite. Et quand on compare **plusieurs** modeles (3 paires ici), le risque de faux
positif grimpe : il faut corriger (Bonferroni : diviser le seuil par le nombre de comparaisons).

In [7]:
# Taille d'effet rank-biserial approximee : standardisation de la difference moyenne par pli.
def effect_size_paired(scores_a, scores_b):
    """Taille d'effet r = d_mean / sqrt(d_mean^2 + var(d)) -- interpretable dans [-1, 1]."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    d_mean = d.mean()
    return d_mean / np.sqrt(d_mean**2 + d.var(ddof=1))

pairs = [
    ("LinearRegression", "RandomForest"),
    ("LinearRegression", "GradientBoosting"),
    ("RandomForest", "GradientBoosting"),
]
n_comparisons = len(pairs)
alpha_bonferroni = 0.05 / n_comparisons
print(f"Seuil Bonferroni pour {n_comparisons} paires : alpha = {alpha_bonferroni:.4f}\n")
for a, b in pairs:
    r_eff = effect_size_paired(cv_results[a], cv_results[b])
    _, _, p_val = corrected_resampled_ttest(cv_results[a], cv_results[b], n_test_fold, n_train_fold)
    sig = "SIGNIFICATIF" if p_val < alpha_bonferroni else "non-significatif"
    print(f"{a:18s} vs {b:18s} : taille d'effet = {r_eff:+.3f}, p = {p_val:.4f} -> {sig}")
print("\nLa taille d'effet distingue 'significatif mais negligeable' (r proche de 0) d'un ecart reel.")

Seuil Bonferroni pour 3 paires : alpha = 0.0167

LinearRegression   vs RandomForest       : taille d'effet = +0.620, p = 0.0594 -> non-significatif
LinearRegression   vs GradientBoosting   : taille d'effet = +0.663, p = 0.0427 -> non-significatif
RandomForest       vs GradientBoosting   : taille d'effet = +0.547, p = 0.0938 -> non-significatif

La taille d'effet distingue 'significatif mais negligeable' (r proche de 0) d'un ecart reel.


## Exercices

### Exercice 1 : un quatrieme modele au banc
Ajoutez `ExtraTreesRegressor(n_estimators=200, random_state=42)` au dictionnaire `models`, relancez
la CV 10-fold, puis comparez-le a GradientBoosting avec le *corrected resampled t-test*.
Le verdict change-t-il apres correction Bonferroni pour 6 paires au lieu de 3 ?

In [8]:
# Exercice 1 : a completer
# from sklearn.ensemble import ExtraTreesRegressor
# ... ajoutez ExtraTrees a cv_results, puis appelez corrected_resampled_ttest vs GradientBoosting.
# N'oubliez pas de mettre a jour n_comparisons et alpha_bonferroni (6 paires -> alpha = 0.05/6).
print("Exercice a completer : banc a 4 modeles avec correction Bonferroni a 6 paires.")

Exercice a completer : banc a 4 modeles avec correction Bonferroni a 6 paires.


### Exercice 2 : IC bootstrap sur RMSE
La fonction `bootstrap_metric_diff` marche avec n'importe quelle metrique. Reprenez-la avec
`mean_squared_error` (RMSE) au lieu de `r2_score`, entre RandomForest et GradientBoosting.
Attention au signe : un RMSE *plus petit* est meilleur.

In [9]:
# Exercice 2 : a completer
# from sklearn.metrics import mean_squared_error
# diffs_rmse, lo, hi = bootstrap_metric_diff(rf, gb, X_te, y_te, mean_squared_error, n_boot=2000)
# Rappelez-vous : pour RMSE, A meilleur que B <=> E[RMSE_A - RMSE_B] < 0.
print("Exercice a completer : IC bootstrap sur la difference de RMSE.")

Exercice a completer : IC bootstrap sur la difference de RMSE.


### Exercice 3 : stabilite de l'IC bootstrap
Combien de reechantillonnages bootstrap (n_boot) faut-il pour que les bornes de l'IC a 95% de la
difference de R2 se stabilisent a +/- 0.005 pres ? Tracez la demi-largeur de l'IC en fonction de
n_boot dans [100, 200, 500, 1000, 2000, 5000].

In [10]:
# Exercice 3 : a completer
# for n_boot in [100, 200, 500, 1000, 2000, 5000]:
#     _, lo, hi = bootstrap_metric_diff(rf, gb, X_te, y_te, r2_score, n_boot=n_boot)
#     ... stockez (hi - lo) / 2, puis tracez-le vs n_boot.
print("Exercice a completer : convergence de la demi-largeur d'IC vs n_boot.")

Exercice a completer : convergence de la demi-largeur d'IC vs n_boot.


## Conclusion

- **Un seul split ne prouve rien** : le classement des modeles varie avec la graine (Section 1).
- **moyenne +/- ecart-type** (Section 2) ameliore la stabilite mais ne constitue pas un test ; le
  chevauchement des barres d'erreur n'est ni necessary ni sufficient pour conclure.
- **Bootstrap** (Section 3) donne un IC robuste sur une metrique ou sur la *difference* entre deux
  modeles, sans hypothese de distribution.
- **Corrected resampled t-test** (Section 4, Nadeau & Bengio 2003) est le test statistiquement
  correct pour comparer deux modeles en k-fold ; le t-test naif sur les plis est anti-conservatoire.
- **Taille d'effet + Bonferroni** (Section 5) separant "significatif" de "important" et controlant
  le risque quand on compare plus de deux modeles.

La lecon transversale : comparer des modeles est une **question statistique**, pas seulement une
question de calcul de metrique. C'est le pendant, pour la modelisation predictive, de ce que la
validite de backtest (PSR, bruit de Sharpe) est pour le trading.

### References
- L. Nadeau and Y. Bengio, *Inference for the Generalization Error*, Machine Learning, 2003.
- A. J. Saltelli et al., *Sensitivity Analysis*, sur la prudence face aux comparaisons multiples.

*Voir aussi : [ML-4](ML-4-Evaluation-Python.ipynb) pour le calcul des metriques de base.